# Sezione 1: Configurazione dell'Ambiente e Download del Testo
Iniziamo scaricando il testo pratico utilizzato nel libro, ovvero il racconto di pubblico dominio "The Verdict" di Edith Wharton.

In [ ]:
import urllib.request
import re
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# URL del file originale sul repository del libro
url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")
file_path = "the-verdict.txt"

if not os.path.exists(file_path):
    print("Scaricamento in corso...")
    urllib.request.urlretrieve(url, file_path)
    print("Scaricamento completato!")
else:
    print("Il file di testo esiste già localmente.")

# Carichiamo il testo in Python
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("GPU attivata")
else:
    device = torch.device("cpu")
    print("MPS non disponibile, utilizzo della CPU.")
    
print("\nLunghezza totale (in caratteri):", len(raw_text))
print("Esempio primi 100 caratteri:\n", raw_text[:100])

Il file di testo esiste già localmente.

Lunghezza totale (in caratteri): 20479
Esempio primi 100 caratteri:
 I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


# Sezione 2: Tokenizzazione Manuale del Testo
Usiamo espressioni regolari per dividere il testo in parole e segni di punteggiatura separati, preservando la capitalizzazione (case-sensitive) per aiutare il modello a riconoscere i nomi propri e la struttura logica della frase.

In [11]:
# Test di tokenizzazione su testo d'esempio
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\\]|--|\\s)', text)
result = [item.strip() for item in result if item.strip()]
print("Esempio semplice di tokenizzazione:")
print(result)

# Applichiamo la tokenizzazione all'intero racconto
preprocessed = re.split(r'([,.:;?_!\"()\\]|--|\\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(f"\nNumero totale di token nel racconto: {len(preprocessed)}")
print("Primi 30 token:")
print(preprocessed[:30])

Esempio semplice di tokenizzazione:
['Hello', ',', 'world', '.', 'Is this', '--', 'a test', '?']

Numero totale di token nel racconto: 1468
Primi 30 token:
['I HAD always thought Jack Gisburn rather a cheap genius', '--', 'though a good fellow enough', '--', 'so it was no great surprise to me to hear that', ',', 'in the height of his glory', ',', 'he had dropped his painting', ',', 'married a rich widow', ',', 'and established himself in a villa on the Riviera', '.', '(', 'Though I rather thought it would have been Rome or Florence', '.', ')', '"', 'The height of his glory', '"', '--', 'that was what the women called it', '.', 'I can hear Mrs', '.', 'Gideon Thwing', '--', 'his last Chicago sitter', '--']


# Sezione 3: Creazione del Vocabolario e Token ID (SimpleTokenizerV1)
Creiamo un vocabolario mappando ogni token unico ad un numero intero sequenziale e definiamo la nostra prima classe Tokenizer per convertire il testo in token ID (encode) e viceversa (decode).

In [14]:
# Estraiamo tutti i token unici e li ordiniamo alfabeticamente
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("Dimensione del vocabolario (token unici):", vocab_size)

# Dizionario del vocabolario (mappa token -> ID intero)
vocab = {token: integer for integer, token in enumerate(all_words)}

# Classe Tokenizer di base
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
        
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!\"()\\]|--|\\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Rimuoviamo gli spazi prima dei segni di punteggiatura principali
        text = re.sub(r'\\s+([,.:;?!\"()\\])', r'\\1', text)
        return text

# Testiamo il Tokenizer V1 su una frase del dataset
tokenizer = SimpleTokenizerV1(vocab)
text_sample = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text_sample)
print("\nToken ID generati:", ids)
print("Testo ricostruito:", tokenizer.decode(ids))

Dimensione del vocabolario (token unici): 625

Token ID generati: [1, 168, 8, 623, 8, 1, 177, 10, 55, 10]
Testo ricostruito: " It's the last he painted , you know , " Mrs . Gisburn said with pardonable pride .


# Sezione 4: Gestione di Parole Sconosciute con Token Speciali (SimpleTokenizerV2)
Se proviamo a usare la versione V1 su testi con parole non incluse nel vocabolario d'addestramento, il codice genera un errore (KeyError). Aggiungiamo quindi due token speciali di contesto:

    <|unk|> (per parole sconosciute o out-of-vocabulary).
    <|endoftext|> (per delimitare documenti separati non legati tra loro).

In [17]:
# Estendiamo il vocabolario
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab_v2 = {token: integer for integer, token in enumerate(all_tokens)}

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
        
    def encode(self, text):
        # Utilizziamo una stringa grezza (r'...') per le espressioni regolari
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # Sostituiamo correttamente i token sconosciuti con "<|unk|>" senza errori di escaping
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Sistemiamo gli spazi prima della punteggiatura
        text = re.sub(r'\s+([,.:;?!"\'])', r'\1', text)
        return text

# Test con parole sconosciute (ad esempio, "palace")
tokenizer_v2 = SimpleTokenizerV2(vocab_v2)
combined_text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
ids_v2 = tokenizer_v2.encode(combined_text)
print("Token ID generati con V2:", ids_v2)
print("Testo ricostruito:", tokenizer_v2.decode(ids_v2))

Token ID generati con V2: [626, 8, 626, 626, 626, 626, 13, 625, 626, 557, 626, 626, 626, 557, 626, 10]
Testo ricostruito: <|unk|>, <|unk|> <|unk|> <|unk|> <|unk|>? <|endoftext|> <|unk|> the <|unk|> <|unk|> <|unk|> the <|unk|>.


# Sezione 5: Tokenizzazione Avanzata Byte Pair Encoding (BPE)
I modelli commerciali evitano il token <|unk|> scomponendo le parole non conosciute in caratteri singoli o sotto-unità di parole comuni tramite l'algoritmo Byte Pair Encoding (BPE). Usiamo la libreria open source tiktoken di OpenAI.

In [19]:
import tiktoken

bpe_tokenizer = tiktoken.get_encoding("gpt2")

# Test BPE su parole sconosciute
text_bpe = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
integers = bpe_tokenizer.encode(text_bpe, allowed_special={"<|endoftext|>"})

print("Token ID generati da BPE:", integers)
print("Testo decodificato:", bpe_tokenizer.decode(integers))

# Esempio pratico: mostriamo la scomposizione della parola inventata "Akwirw ier"
unknown_word = "Akwirw ier"
word_ids = bpe_tokenizer.encode(unknown_word)
print(f"\nScomposizione di '{unknown_word}':")
for i in word_ids:
    print(f"Token ID {i} -> '{bpe_tokenizer.decode([i])}'")

Token ID generati da BPE: [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Testo decodificato: Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.

Scomposizione di 'Akwirw ier':
Token ID 33901 -> 'Ak'
Token ID 86 -> 'w'
Token ID 343 -> 'ir'
Token ID 86 -> 'w'
Token ID 220 -> ' '
Token ID 959 -> 'ier'


# Sezione 6: Dataset e DataLoader con Finestra Scorrevole
Per addestrare il modello alla previsione del token successivo, estraiamo coppie di input-target con un approccio a finestra scorrevole (sliding window). Il target y è semplicemente l'input x spostato di un token in avanti.

In [20]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)
        
        # Generazione delle coppie input-target con finestra scorrevole
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
            
    def __len__(self):
        return len(self.input_ids)
        
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True, 
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

# Testiamo il DataLoader con lotti di batch_size=8 e max_length=4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs shape:", inputs.shape)
print("Targets shape:", targets.shape)

Inputs shape: torch.Size([8, 4])
Targets shape: torch.Size([8, 4])


# Sezione 7: Creazione di Token Embeddings e Positional Embeddings
Nello step finale della pipeline, gli ID dei token interi vengono convertiti in vettori densi nello spazio vettoriale continuo del modello. Ad essi sommiamo gli embeddings posizionali per fornire l'informazione sull'ordine temporale della sequenza.

In [26]:
vocab_size = 50257
output_dim = 256
max_length = 4

# 1. Creiamo i layer di embedding
token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = nn.Embedding(max_length, output_dim)

# 2. Spostiamo i layer sul device selezionato (MPS o CUDA)
token_embedding_layer = token_embedding_layer.to(device)
pos_embedding_layer = pos_embedding_layer.to(device)

inputs = inputs.to(device)

# 3. Calcolo degli embeddings per i token (ora eseguito interamente su GPU/MPS)
token_embeddings = token_embedding_layer(inputs)
print("Shape degli embeddings dei token:", token_embeddings.shape) 
# Output atteso: torch.Size([8, 4, 256])

# 4. Creazione degli indici posizionali direttamente sul device corretto
positions = torch.arange(max_length, device=device)
pos_embeddings = pos_embedding_layer(positions)
print("Shape degli embeddings posizionali:", pos_embeddings.shape)
# Output atteso: torch.Size([4, 256])

# 5. Somma finale degli embeddings (con broadcasting del batch)
input_embeddings = token_embeddings + pos_embeddings
print("Rappresentazione finale pronta per la rete neurale:", input_embeddings.shape)
# Output atteso: torch.Size([8, 4, 256])

Shape degli embeddings dei token: torch.Size([8, 4, 256])
Shape degli embeddings posizionali: torch.Size([4, 256])
Rappresentazione finale pronta per la rete neurale: torch.Size([8, 4, 256])
